# adsb.lol flights ingest

Consumes the `flights-positions` topic produced by `producer_flights.py` and
appends one row per aircraft per snapshot to Delta.

Serverless (Databricks Free Edition) only supports the `availableNow` trigger.
The query drains the offsets available at start, then stops. `failOnDataLoss`
is false because Aiven retention is short: if the job has not run for longer
than retention, offsets expire and the query must skip the gap instead of
failing. `maxOffsetsPerTrigger` caps the first catch-up batch.

In [ ]:
# Databricks notebook source
from pyspark.sql.functions import col, from_json, to_timestamp
from pyspark.sql.types import DoubleType, LongType, StringType, StructType

# Message contract produced by producer_flights.py.
schema = StructType() \
    .add("timestamp", StringType()) \
    .add("icao24", StringType()) \
    .add("callsign", StringType()) \
    .add("country", StringType()) \
    .add("lat", DoubleType()) \
    .add("lon", DoubleType()) \
    .add("last_update_ms", LongType()) \
    .add("snapshot_id", StringType()) \
    .add("fetched_at", StringType()) \
    .add("source", StringType())

aiven_password = dbutils.secrets.get(scope="kafka-scope", key="aiven-password")
jaas_config = f'kafkashaded.org.apache.kafka.common.security.scram.ScramLoginModule required username="avnadmin" password="{aiven_password}";'

kafka_df = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka-26289383-md-f1a0.l.aivencloud.com:26452")
    .option("subscribe", "flights-positions")
    # `earliest` only applies when the checkpoint is new; after that the
    # checkpoint owns the offsets. It gives the first run the retained history.
    .option("startingOffsets", "earliest")
    .option("maxOffsetsPerTrigger", "50000")
    .option("failOnDataLoss", "false")
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "SCRAM-SHA-256")
    .option("kafka.sasl.jaas.config", jaas_config)
    .option("kafka.ssl.truststore.location", "/Volumes/workspace/default/energy_realtime/aiven_ca.pem")
    .option("kafka.ssl.truststore.type", "PEM")
    .load()
)

parsed_df = kafka_df.select(
    from_json(col("value").cast("string"), schema).alias("data")
).select("data.*")

clean_df = (parsed_df
    .where(col("icao24").isNotNull() & col("lat").isNotNull() & col("lon").isNotNull())
    .select(
        to_timestamp(col("timestamp")).alias("observed_at"),
        "icao24",
        "callsign",
        "country",
        "lat",
        "lon",
        "last_update_ms",
        to_timestamp(col("snapshot_id")).alias("snapshot_id"),
        to_timestamp(col("fetched_at")).alias("fetched_at"),
        "source",
    ))

query = (clean_df.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", "/Volumes/workspace/default/energy_realtime/checkpoints-flights")
    .toTable("flights_positions"))

query.awaitTermination()